# Analyze Eclipse Petrophysical Properties

In [1]:
# Import Python libraries
!pip install -q -U kaleido==0.2.1 -q -U skimpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.2.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
ju

In [2]:
import os
from google.colab import drive
from pathlib import Path

# Mount Google Drive (run once per session)
drive.mount("/content/drive")

# Data directory (adjust if you move the notebook)
source = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/")
dest = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne/")

# Create directory if it does not exist
os.makedirs(dest, exist_ok=True)

print("Source directory:", source)
print("Output directory:", dest)

Mounted at /content/drive
Source directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO
Output directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne


In [3]:
from pathlib import Path

# All entries (files + subdirs)
for p in source.iterdir():
    print(p)

# Only files
#for p in source.iterdir():
#    if p.is_file():
#        print(p)

/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/SWINITIAL.INC
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/PORO_0704.prop
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/EXTRA_REG.inc
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/FLUXNUM_0704.prop
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/MULTREGT_D_27.prop
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/PERM_0704.prop
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/NTG_0704.prop
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/MULTZ_HM_1.INC
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PETRO/MULTZ_JUN_05_MOD.INC
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/P

## 1.0 2D Map Plots

Here is a full, self‑contained script that:

1. Reads the main property files

2. Builds a 3D xarray.Dataset

3. Plots PORO and FLUXNUM maps with equal I–J scale (square cells)

In [4]:
from pathlib import Path
import numpy as np
import xarray as xr
import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"
import plotly.express as px

# ----------------------------------------------------------------------
# 1. Grid dimensions from SPECGRID in IRAP_1005.GRDECL
# ----------------------------------------------------------------------
NX, NY, NZ = 46, 112, 22   # from SPECGRID 46 112 22 1 F
n_cells = NX * NY * NZ

base = Path(".")

# ----------------------------------------------------------------------
# 2. Helper to read flat Eclipse property / INC files
# ----------------------------------------------------------------------
def read_prop_flat(path, n_expected):
    vals = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("--"):
                continue
            if line == "/":
                break
            line = line.replace("/", " ")
            for tok in line.split():
                try:
                    vals.append(float(tok))
                except ValueError:
                    pass
    if len(vals) != n_expected:
        print(f"Warning: {path.name} has {len(vals)} values, expected {n_expected}")
    return np.array(vals[:n_expected], dtype=float)

# ----------------------------------------------------------------------
# 3. Read scalar properties
# ----------------------------------------------------------------------
poro    = read_prop_flat(source / "PORO_0704.prop",    n_cells)  # porosity
perm    = read_prop_flat(source / "PERM_0704.prop",    n_cells)  # permeability (likely Kx)
ntg     = read_prop_flat(source / "NTG_0704.prop",     n_cells)  # net-to-gross
fluxnum = read_prop_flat(source / "FLUXNUM_0704.prop", n_cells)  # region index
fipnum  = read_prop_flat(source / "FIPNUM_0704.prop",  n_cells)  # FIP region
eqlnum  = read_prop_flat(source / "EQLNUM_0704.prop",  n_cells)  # PVT region

# Optional: MULTZ / SWINITIAL fields, similar pattern:
# multz   = read_prop_flat(source / "MULTZ_HM_1.INC",   n_cells)
# swinit  = read_prop_flat(source / "SWINITIAL.INC",    n_cells)

shape = (NZ, NY, NX)
poro_3d    = poro.reshape(shape)
perm_3d    = perm.reshape(shape)
ntg_3d     = ntg.reshape(shape)
fluxnum_3d = fluxnum.reshape(shape)
fipnum_3d  = fipnum.reshape(shape)
eqlnum_3d  = eqlnum.reshape(shape)

# ----------------------------------------------------------------------
# 4. Build xarray Dataset
# ----------------------------------------------------------------------
k = np.arange(1, NZ+1)
j = np.arange(1, NY+1)
i = np.arange(1, NX+1)

ds = xr.Dataset(
    {
        "PORO":    (("k", "j", "i"), poro_3d),
        "PERM":    (("k", "j", "i"), perm_3d),
        "NTG":     (("k", "j", "i"), ntg_3d),
        "FLUXNUM": (("k", "j", "i"), fluxnum_3d),
        "FIPNUM":  (("k", "j", "i"), fipnum_3d),
        "EQLNUM":  (("k", "j", "i"), eqlnum_3d),
    },
    coords={"k": k, "j": j, "i": i},
)
print(ds)

# ----------------------------------------------------------------------
# 5. Example stats
# ----------------------------------------------------------------------
summary = {}
for v in ["PORO", "PERM", "NTG"]:
    arr = ds[v].values
    summary[v] = dict(
        min=float(np.nanmin(arr)),
        max=float(np.nanmax(arr)),
        mean=float(np.nanmean(arr)),
    )
print("Summary stats:", summary)

# ----------------------------------------------------------------------
# 6. Plotly maps with equal I–J scale
# ----------------------------------------------------------------------
k_mid = NZ // 2 + 1   # mid-layer (1-based)

# PORO map at mid-layer
poro_mid = ds["PORO"].sel(k=k_mid)
fig_poro = px.imshow(
    poro_mid.values,
    origin="lower",
    color_continuous_scale="Viridis",
    labels=dict(color="PORO"),
)
fig_poro.update_layout(
    title=f"PORO at k={k_mid}",
    xaxis_title="i",
    yaxis_title="j",
    width=500,    # reduced width
    height=600,   # optional
)
fig_poro.update_yaxes(scaleanchor="x", constrain="domain")
fig_poro.update_xaxes(constrain="domain")
fig_poro.show()

# PERM map at mid-layer
perm_mid = ds["PERM"].sel(k=k_mid)
fig_perm = px.imshow(
    perm_mid.values,
    origin="lower",
    color_continuous_scale="Viridis",
    labels=dict(color="PERM"),
)
fig_perm.update_layout(
    title=f"PERM at k={k_mid}",
    xaxis_title="i",
    yaxis_title="j",
    width=500,    # reduced width
    height=600,   # optional
)
fig_perm.update_yaxes(scaleanchor="x", constrain="domain")
fig_perm.update_xaxes(constrain="domain")
fig_perm.show()

# FLUXNUM map at top layer
flux_top = ds["FLUXNUM"].sel(k=1)
fig_flux = px.imshow(
    flux_top.values,
    origin="lower",
    color_continuous_scale="Turbo",
    labels=dict(color="FLUXNUM"),
)
fig_flux.update_layout(
    title="FLUXNUM at k=1",
    xaxis_title="i",
    yaxis_title="j",
    width=500,    # reduced width
    height=600,   # optional
)
fig_flux.update_yaxes(scaleanchor="x", constrain="domain")
fig_flux.update_xaxes(constrain="domain")

# Define title here to be accessible for saving
plot_title = "PORO-PERM-FLUXNUM"

# Ensure dest is a Path object for file operations
dest = Path(dest) # Explicitly convert dest to Path

plotly.offline.plot(fig_flux, filename = str(dest / (plot_title + '.html'))) # Changed plotly.offline.plot to pyo.plot and title to plot_title
fig_flux.write_image(str(dest / (plot_title + '.png')), scale=3.125)

fig_flux.show()

<xarray.Dataset> Size: 5MB
Dimensions:  (k: 22, j: 112, i: 46)
Coordinates:
  * k        (k) int64 176B 1 2 3 4 5 6 7 8 9 10 ... 14 15 16 17 18 19 20 21 22
  * j        (j) int64 896B 1 2 3 4 5 6 7 8 ... 105 106 107 108 109 110 111 112
  * i        (i) int64 368B 1 2 3 4 5 6 7 8 9 10 ... 38 39 40 41 42 43 44 45 46
Data variables:
    PORO     (k, j, i) float64 907kB 0.3054 0.3029 0.2996 ... 0.2622 0.2569
    PERM     (k, j, i) float64 907kB 213.5 221.0 225.2 ... 1.034e+03 912.7
    NTG      (k, j, i) float64 907kB 0.9779 0.9788 0.9799 ... 0.8113 0.8087
    FLUXNUM  (k, j, i) float64 907kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    FIPNUM   (k, j, i) float64 907kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    EQLNUM   (k, j, i) float64 907kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Summary stats: {'PORO': {'min': 0.0, 'max': 0.349999994, 'mean': 0.20432019623411649}, 'PERM': {'min': 0.0, 'max': 3996.54761, 'mean': 290.7676313575064}, 'NTG': {'min': 0.0, 'max': 1.0, 'mean': 0.7

This now visualizes PORO, PERM, and FLUXNUM slices with I and J on the same scale for each map.

In [5]:
summary

{'PORO': {'min': 0.0, 'max': 0.349999994, 'mean': 0.20432019623411649},
 'PERM': {'min': 0.0, 'max': 3996.54761, 'mean': 290.7676313575064},
 'NTG': {'min': 0.0, 'max': 1.0, 'mean': 0.7458668235051225}}

In [7]:
ds

<xarray.Dataset> Size: 5MB
Dimensions:  (k: 22, j: 112, i: 46)
Coordinates:
  * k        (k) int64 176B 1 2 3 4 5 6 7 8 9 10 ... 14 15 16 17 18 19 20 21 22
  * j        (j) int64 896B 1 2 3 4 5 6 7 8 ... 105 106 107 108 109 110 111 112
  * i        (i) int64 368B 1 2 3 4 5 6 7 8 9 10 ... 38 39 40 41 42 43 44 45 46
Data variables:
    PORO     (k, j, i) float64 907kB 0.3054 0.3029 0.2996 ... 0.2622 0.2569
    PERM     (k, j, i) float64 907kB 213.5 221.0 225.2 ... 1.034e+03 912.7
    NTG      (k, j, i) float64 907kB 0.9779 0.9788 0.9799 ... 0.8113 0.8087
    FLUXNUM  (k, j, i) float64 907kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    FIPNUM   (k, j, i) float64 907kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    EQLNUM   (k, j, i) float64 907kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0

In [8]:
from skimpy import skim

# Skimpy’s main function skim(df) produces rich summary statistics for every column in a DataFrame
# Convert the xarray.Dataset to a pandas.DataFrame
df = ds.to_dataframe()
skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 113344 │ │ float64     │ 6     │                                                          │
│ │ Number of columns │ 6      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column     ┃ NA   ┃ NA %   ┃ mean     ┃ sd       ┃ p0  ┃ p25      ┃ p50      ┃ p75      ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │    PORO    │    0 │      0 │   0.2043 │   0.0811 │   0 │   0.1987 │   0.2287 │   0.2491 │   0.35 │  ▂ ▁▇█  │  │
│ │    PERM    │    0 │      0 │    290.8 │    428.7 │   0 │    37.26 │    134.9 │    356.9 │   3997 │   █▁    │  │
│ │    NTG     │    0 │      0 │   0.7459 │   0.3201 │   0 │   0.6707 │    0.891 │   0.9724 │      1 │ ▂ ▁▁▂█  │  │
│ │  FLUXNUM   │    0 │      0 │    3.351 │    5.028 │   0 │        0 │        0 │        7 │     20 │  █▁▁▁   │  │
│ │   FIPNUM   │    0 │      0 │    3.204 │    4.643 │   0 │        0 │        0 │        7 │     16 │  █▁ ▁▁  │  │
│ │   EQLNUM   │    0 │      0 │    1.726 │    2.273 │   0 │        0 │        0 │        5 │      5 │ █   ▁▄  │  │
│ └────────────┴──────┴────────┴──────────┴──────────┴─────┴──────────┴──────────┴──────────┴────────┴─────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

In [15]:
df

PORO         PERM       NTG  FLUXNUM  FIPNUM  EQLNUM
k  j   i                                                           
1  1   1   0.305408   213.478149  0.977852      0.0     0.0     0.0
       2   0.302881   221.009415  0.978850      0.0     0.0     0.0
       3   0.299644   225.161331  0.979851      0.0     0.0     0.0
       4   0.296977   231.829575  0.980839      0.0     0.0     0.0
       5   0.293957   235.915298  0.981798      0.0     0.0     0.0
...             ...          ...       ...      ...     ...     ...
22 112 42  0.266288  1172.968140  0.817958      0.0     0.0     0.0
       43  0.265653  1143.479980  0.815955      0.0     0.0     0.0
       44  0.264730  1106.846920  0.813720      0.0     0.0     0.0
       45  0.262164  1034.245240  0.811280      0.0     0.0     0.0
       46  0.256903   912.741760  0.808667      0.0     0.0     0.0

[113344 rows x 6 columns]

In [9]:
# Export to CSV
df.to_csv(dest / "df_output.csv", index=False)

# Export to Excel Single sheet
df.to_excel(dest / "df_output.xlsx", index=False, freeze_panes=(1,0))

## 2.0 3D Map Plots

In [11]:
import plotly.graph_objects as go

# Assume ds has dimensions (k, j, i) or similar
poro = ds["PORO"]          # full 3D porosity
perm = ds["PERM"]          # full 3D permeability
flux = ds["FLUXNUM"]       # full 3D fluxnum (we’ll use top layer)

NZ = poro.sizes["k"]
NY = poro.sizes["j"]
NX = poro.sizes["i"]

# Build i, j, k index arrays
import numpy as np

k_vals = np.arange(1, NZ + 1)
j_vals = np.arange(1, NY + 1)
i_vals = np.arange(1, NX + 1)

K, J, I = np.meshgrid(k_vals, j_vals, i_vals, indexing="ij")

# -----------------------------
# 3D PORO volume
# -----------------------------
fig_poro_3d = go.Figure(
    data=go.Volume(
        x=I.flatten(),
        y=J.flatten(),
        z=K.flatten(),
        value=poro.values.flatten(),
        opacity=0.1,         # lower for more transparency
        surface_count=10,
        colorscale="Viridis",
        colorbar=dict(title="PORO"),
    )
)

fig_poro_3d.update_layout(
    title="3D PORO Volume",
    scene=dict(
        xaxis_title="i",
        yaxis_title="j",
        zaxis_title="k",
        aspectmode="data",
    ),
    width=800,
    height=700,
)

fig_poro_3d.show()

# -----------------------------
# 3D PERM volume
# -----------------------------
fig_perm_3d = go.Figure(
    data=go.Volume(
        x=I.flatten(),
        y=J.flatten(),
        z=K.flatten(),
        value=perm.values.flatten(),
        opacity=0.1,
        surface_count=10,
        colorscale="Viridis",
        colorbar=dict(title="PERM"),
    )
)

fig_perm_3d.update_layout(
    title="3D PERM Volume",
    scene=dict(
        xaxis_title="i",
        yaxis_title="j",
        zaxis_title="k",
        aspectmode="data",
    ),
    width=800,
    height=700,
)

fig_perm_3d.show()

# -----------------------------
# FLUXNUM as 3D surface at top layer
# -----------------------------
flux_top = flux.sel(k=1)

fig_flux_3d = go.Figure(
    data=go.Surface(
        x=i_vals,
        y=j_vals,
        z=np.ones_like(flux_top.values) * 1,   # z = 1 (top layer)
        surfacecolor=flux_top.values,
        colorscale="Turbo",
        colorbar=dict(title="FLUXNUM"),
    )
)

fig_flux_3d.update_layout(
    title="FLUXNUM at k=1 (3D surface)",
    scene=dict(
        xaxis_title="i",
        yaxis_title="j",
        zaxis_title="k",
        aspectmode="data",
    ),
    width=800,
    height=700,
)

# Save flux figure as before
plot_title = "PORO-PERM-FLUXNUM-3D"
dest = Path(dest)

import plotly.offline as pyo
pyo.plot(fig_flux_3d, filename=str(dest / (plot_title + ".html")))
fig_flux_3d.write_image(str(dest / (plot_title + ".png")), scale=3.125)

fig_flux_3d.show()

Output hidden; open in https://colab.research.google.com to view.

In [12]:
import plotly.colors as pc

# List of all colorscale names (continuous + discrete, incl. *_r variants)
all_scales = pc.named_colorscales()
print(len(all_scales))
print(all_scales)

94
['aggrnyl', 'agsunset', 'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'darkmint', 'electric', 'emrld', 'gnbu', 'greens', 'greys', 'hot', 'inferno', 'jet', 'magenta', 'magma', 'mint', 'orrd', 'oranges', 'oryel', 'peach', 'pinkyl', 'plasma', 'plotly3', 'pubu', 'pubugn', 'purd', 'purp', 'purples', 'purpor', 'rainbow', 'rdbu', 'rdpu', 'redor', 'reds', 'sunset', 'sunsetdark', 'teal', 'tealgrn', 'turbo', 'viridis', 'ylgn', 'ylgnbu', 'ylorbr', 'ylorrd', 'algae', 'amp', 'deep', 'dense', 'gray', 'haline', 'ice', 'matter', 'solar', 'speed', 'tempo', 'thermal', 'turbid', 'armyrose', 'brbg', 'earth', 'fall', 'geyser', 'prgn', 'piyg', 'picnic', 'portland', 'puor', 'rdgy', 'rdylbu', 'rdylgn', 'spectral', 'tealrose', 'temps', 'tropic', 'balance', 'curl', 'delta', 'oxy', 'edge', 'hsv', 'icefire', 'phase', 'twilight', 'mrybm', 'mygbm']


In [13]:
import plotly.express as px

# Show all sequential names
dir(px.colors.sequential)

# Show all diverging names
dir(px.colors.diverging)

# Show all cyclical names
dir(px.colors.cyclical)

# View swatches of all continuous scales
px.colors.sequential.swatches_continuous().show()
px.colors.diverging.swatches_continuous().show()
px.colors.cyclical.swatches_continuous().show()

In [14]:
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import plotly.offline as pyo

# Assume ds has dimensions (k, j, i) or similar
poro = ds["PORO"]
perm = ds["PERM"]
flux = ds["FLUXNUM"]

NZ = poro.sizes["k"]
NY = poro.sizes["j"]
NX = poro.sizes["i"]

k_vals = np.arange(1, NZ + 1)
j_vals = np.arange(1, NY + 1)
i_vals = np.arange(1, NX + 1)

K, J, I = np.meshgrid(k_vals, j_vals, i_vals, indexing="ij")

dest = Path(dest)

# 3D PORO volume
fig_poro_3d = go.Figure(
    data=go.Volume(
        x=I.flatten(),
        y=J.flatten(),
        z=K.flatten(),
        value=poro.values.flatten(),
        opacity=0.1,
        surface_count=10,
        colorscale="Turbo",
        colorbar=dict(title="PORO"),
    )
)
fig_poro_3d.update_layout(
    title="3D PORO Volume",
    scene=dict(
        xaxis_title="i",
        yaxis_title="j",
        zaxis_title="k",
        aspectmode="data",
    ),
    width=800,
    height=700,
)

plot_title = "PORO-3D"
pyo.plot(fig_poro_3d, filename=str(dest / f"{plot_title}.html"))
fig_poro_3d.write_image(str(dest / f"{plot_title}.png"), scale=3.125)
fig_poro_3d.show()   # shown once

# 3D PERM volume
fig_perm_3d = go.Figure(
    data=go.Volume(
        x=I.flatten(),
        y=J.flatten(),
        z=K.flatten(),
        value=perm.values.flatten(),
        opacity=0.1,
        surface_count=10,
        colorscale="Turbo",
        colorbar=dict(title="PERM"),
    )
)
fig_perm_3d.update_layout(
    title="3D PERM Volume",
    scene=dict(
        xaxis_title="i",
        yaxis_title="j",
        zaxis_title="k",
        aspectmode="data",
    ),
    width=800,
    height=700,
)

plot_title = "PERM-3D"
pyo.plot(fig_perm_3d, filename=str(dest / f"{plot_title}.html"))
fig_perm_3d.write_image(str(dest / f"{plot_title}.png"), scale=3.125)
fig_perm_3d.show()

# FLUXNUM as 3D surface at top layer
flux_top = flux.sel(k=1)

fig_flux_3d = go.Figure(
    data=go.Surface(
        x=i_vals,
        y=j_vals,
        z=np.ones_like(flux_top.values) * 1,
        surfacecolor=flux_top.values,
        colorscale="Turbo",
        colorbar=dict(title="FLUXNUM"),
    )
)
fig_flux_3d.update_layout(
    title="FLUXNUM at k=1 (3D surface)",
    scene=dict(
        xaxis_title="i",
        yaxis_title="j",
        zaxis_title="k",
        aspectmode="data",
    ),
    width=800,
    height=700,
)

plot_title = "PORO-PERM-FLUXNUM-3D"
pyo.plot(fig_flux_3d, filename=str(dest / f"{plot_title}.html"))
fig_flux_3d.write_image(str(dest / f"{plot_title}.png"), scale=3.125)
fig_flux_3d.show()

Output hidden; open in https://colab.research.google.com to view.